# Import Libraries

In [16]:
import pandas as pd
import numpy as np
import h5py
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, cross_val_score
import matplotlib.pyplot as plt
import os
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Conv1D, GlobalMaxPooling1D, SimpleRNN, LSTM, Input, BatchNormalization, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
import torch
from collections import Counter

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [3]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

# Pre-Processing

### Import Data

In [4]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}
label_mapper = {
    'BT1' : 'knowledge',
    'BT2' : 'comprehension',
    'BT3' : 'application',
    'BT4' : 'analysis',
    'BT5' : 'synthesis',
    'BT6' : 'evaluation'
}


# Load dataset
df = pd.DataFrame()
for i in [2,3,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df])

# Apply preprocessing
df['label'] = df['label'].replace(label_mapper)
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

df['processed_question'] = df['question'].apply(preprocess_text)
df['processed_question'] = [''.join(text) for text in df['processed_question']]

## Tokenize

BERT model is used to retain semantics
,Execute if BERT Embedding needed

In [52]:
# Load pre-trained BERT (TensorFlow version)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = TFBertModel.from_pretrained('bert-base-uncased')

# Tokenize and encode
inputs = tokenizer(df['question'].tolist(), return_tensors='tf', padding=True, truncation=True)
outputs = model(**inputs)

# Extract embeddings (CLS token for sentence representation)
sentence_embedding = outputs.last_hidden_state[:, 0, :]
print("BERT Sentence Embedding Shape:", sentence_embedding.shape)

embedding = pd.DataFrame(sentence_embedding.numpy())

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

BERT Sentence Embedding Shape: (2512, 768)


### Classical Tokenizer

In [34]:
num_size = 20000

tokenizer = Tokenizer(num_words=num_size, oov_token='<OOV>')
tokenizer.fit_on_texts(df['processed_question'])
sequences = tokenizer.texts_to_sequences(df['processed_question'])

In [35]:
max_len = 0

for seq in sequences:
    max_len = max(max_len , len(seq))

print(max_len)

data = pad_sequences(sequences, maxlen=max_len)

21


# Modelling

## Basic Machine Learning

In [36]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = df['label'].map(y_mapper)

In [37]:
x , y = data , to_categorical(y_mapped , 6)
x_train , x_test , y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=34 , stratify= y_mapped)

In [38]:
EMBEDDING_DIM = 28
word_index = tokenizer.word_index
VOCAB_SIZE = len(word_index) + 1

## 1D CNN

In [57]:
cnn_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(VOCAB_SIZE, EMBEDDING_DIM),

    Conv1D(128, 5, activation='gelu', padding= 'same'),
    BatchNormalization(),
    GlobalMaxPooling1D(),
    Dropout(0.3),

    Dense(64, activation='sigmoid'),
    Dropout(0.4),

    Dense(6, activation='softmax')
])
cnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

cnn_model.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_20 (Embedding)        │ (None, 21, 28)         │        71,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_27 (Conv1D)              │ (None, 21, 128)        │        18,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 21, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_17         │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_41 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_42 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 99,166 (387.37 KB)

 Trainable params: 98,910 (386.37 KB)

 Non-trainable params: 256 (1.00 KB)

In [58]:
cnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1660 - loss: 2.0493 - val_accuracy: 0.1642 - val_loss: 1.8407
Epoch 2/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1802 - loss: 1.9275 - val_accuracy: 0.1642 - val_loss: 1.8254
Epoch 3/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1769 - loss: 1.8720 - val_accuracy: 0.1642 - val_loss: 1.8160
Epoch 4/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1927 - loss: 1.8538 - val_accuracy: 0.1642 - val_loss: 1.8107
Epoch 5/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2179 - loss: 1.7967 - val_accuracy: 0.1642 - val_loss: 1.8049
Epoch 6/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2108 - loss: 1.8020 - val_accuracy: 0.1592 - val_loss: 1.7991
Epoch 7/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2503 - loss: 1.7693 - val_accuracy: 0.1642 - val_loss: 1.7891
Epoch 8/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2426 - loss: 1.7610 - val_accuracy: 0.1692 - v

In [59]:
loss, acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 82.50%


## RNN

In [89]:
rnn_model = Sequential([
        Input(shape=(max_len,)),
        Embedding(VOCAB_SIZE, EMBEDDING_DIM),

        SimpleRNN(64, activation= 'tanh'),
        Dropout(0.3),
        
        Dense(32, activation='relu'),
        Dropout(0.4),
        
        Dense(6, activation='softmax')
    ])

rnn_model.compile(loss='categorical_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])

rnn_model.summary()

Model: "sequential_32"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_32 (Embedding)        │ (None, 21, 28)         │        71,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_14 (SimpleRNN)       │ (None, 64)             │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_65 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_66 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_65 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,190 (313.24 KB)

 Trainable params: 80,190 (313.24 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2)

Epoch 1/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1546 - loss: 1.8003 - val_accuracy: 0.1692 - val_loss: 1.7936
Epoch 2/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1649 - loss: 1.7923 - val_accuracy: 0.1965 - val_loss: 1.7902
Epoch 3/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1794 - loss: 1.7865 - val_accuracy: 0.1891 - val_loss: 1.7874
Epoch 4/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1939 - loss: 1.7826 - val_accuracy: 0.2040 - val_loss: 1.7843
Epoch 5/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2213 - loss: 1.7727 - val_accuracy: 0.2164 - val_loss: 1.7810
Epoch 6/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2261 - loss: 1.7632 - val_accuracy: 0.2488 - val_loss: 1.7737
Epoch 7/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2620 - loss: 1.7546 - val_accuracy: 0.2562 - val_loss: 1.7651
Epoch 8/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2916 - loss: 1.7422 - val_accuracy: 0.2463 - v

In [91]:
loss, acc = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc * 100:.2f}%")

Test Accuracy: 72.17%


## LSTM

## Transformer